# UniClubs — Hybrid Recommendation Model Training

Trains a TFRS two-tower model and saves 4 sub-models for hybrid inference.

Architecture:
- `user_collab`: user_id string → learned embedding (collaborative)
- `user_content`: 12-dim feature vector → content embedding
- `event_collab`: event_id string → learned embedding (collaborative)
- `event_content`: 13-dim feature vector → content embedding

At inference: `Score = α × ContentScore + β × CollaborativeScore`

In [155]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print('TF:', tf.__version__, '| TFRS:', tfrs.__version__)

TF: 2.16.2 | TFRS: v0.7.7


## 1. Constants & Vocabulary

In [156]:
EMBEDDING_DIM   = 32
EPOCHS          = 50
BATCH_SIZE      = 64

CATEGORIES = ['tech', 'sports', 'arts', 'academic', 'social']
ALL_TAGS   = ['workshop', 'seminar', 'competition', 'social', 'networking', 'guest_speaker']
INTERESTS  = ['AI', 'Mobile Dev', 'Coding', 'Sports', 'Design', 'Art', 'Photography', 'Music', 'Academic']
INTEREST_TO_CATEGORY = {
    'AI': 'tech', 'Mobile Dev': 'tech', 'Coding': 'tech',
    'Sports': 'sports',
    'Design': 'arts', 'Art': 'arts', 'Photography': 'arts', 'Music': 'arts',
    'Academic': 'academic',
}

# Dimensions:
# user_content_vector = interest_vec(5) + attended_vec(5) + participation_freq(1) + account_age(1) = 12
# event_content_vector = category_onehot(5) + tag_vec(6) + time_of_day(1) + day_of_week(1) = 13
USER_CONTENT_DIM = 12
EVENT_CONTENT_DIM = 13

OUTPUT_DIR = '../ml_models/recommendation_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

Output dir: ../ml_models/recommendation_model


## 2. Synthetic Data Generation

Simulates realistic user-event interactions based on interest-category alignment.
Replace this with real Firestore data once enough interactions are collected.

In [157]:
N_USERS        = 150
N_EVENTS       = 250
N_INTERACTIONS = 3000

users = [
    {
        'user_id': f'user_{i:04d}',
        'interests': random.sample(INTERESTS, random.randint(1, 4)),
        'account_age_days': random.randint(10, 730),
    }
    for i in range(N_USERS)
]

events = [
    {
        'event_id': f'event_{i:04d}',
        'category': random.choice(CATEGORIES),
        'tags': random.sample(ALL_TAGS, random.randint(1, 3)),
        'time_of_day': random.randint(8, 20),
        'day_of_week': random.randint(0, 6),
    }
    for i in range(N_EVENTS)
]

user_dict  = {u['user_id']: u for u in users}
event_dict = {e['event_id']: e for e in events}

# Give each user 3–5 favourite events from their interest categories.
# 60% of interactions go to favourites → strong per-event collaborative signal.
# 30% go to any matching category event → general category preference.
# 10% random → noise/exploration.
user_favorites = {}
for user in users:
    preferred = {INTEREST_TO_CATEGORY.get(i) for i in user['interests'] if INTEREST_TO_CATEGORY.get(i)}
    matching  = [e for e in events if e['category'] in preferred]
    pool      = matching if matching else events
    user_favorites[user['user_id']] = random.sample(pool, min(5, len(pool)))

interactions = []
for _ in range(N_INTERACTIONS):
    user      = random.choice(users)
    preferred = {INTEREST_TO_CATEGORY.get(i) for i in user['interests'] if INTEREST_TO_CATEGORY.get(i)}
    matching  = [e for e in events if e['category'] in preferred]
    rand      = random.random()
    if rand < 0.6:
        event = random.choice(user_favorites[user['user_id']])
    elif rand < 0.9:
        event = random.choice(matching if matching else events)
    else:
        event = random.choice(events)
    interactions.append({'user_id': user['user_id'], 'event_id': event['event_id']})

interactions_df = pd.DataFrame(interactions).drop_duplicates()
print(f'Users: {len(users)}, Events: {len(events)}, Interactions: {len(interactions_df)}')

Users: 150, Events: 250, Interactions: 1770


## 3. Feature Builders

Matches the feature engineering described in CLAUDE.md:
- **User features**: interests (multi-hot over categories), attended categories (multi-hot), participation frequency, account age
- **Event features**: category (one-hot), tags (multi-hot), time of day, day of week

In [158]:
def build_user_content_vector(user, attended_categories, participation_count):
    """12-dim float32 vector: interest_vec(5) + attended_vec(5) + participation(1) + age(1)"""
    interest_vec = np.zeros(len(CATEGORIES), dtype=np.float32)
    for interest in user['interests']:
        cat = INTEREST_TO_CATEGORY.get(interest)
        if cat in CATEGORIES:
            interest_vec[CATEGORIES.index(cat)] = 1.0

    attended_vec = np.zeros(len(CATEGORIES), dtype=np.float32)
    for cat in attended_categories:
        if cat in CATEGORIES:
            attended_vec[CATEGORIES.index(cat)] = 1.0

    participation_freq = np.array([min(participation_count / 20.0, 1.0)], dtype=np.float32)
    account_age        = np.array([min(user['account_age_days'] / 730.0, 1.0)], dtype=np.float32)
    return np.concatenate([interest_vec, attended_vec, participation_freq, account_age])


def build_event_content_vector(event):
    """13-dim float32 vector: category_onehot(5) + tag_vec(6) + time(1) + day(1)"""
    cat_onehot = np.zeros(len(CATEGORIES), dtype=np.float32)
    if event['category'] in CATEGORIES:
        cat_onehot[CATEGORIES.index(event['category'])] = 1.0

    tag_vec = np.zeros(len(ALL_TAGS), dtype=np.float32)
    for tag in event.get('tags', []):
        if tag in ALL_TAGS:
            tag_vec[ALL_TAGS.index(tag)] = 1.0

    time_of_day  = np.array([event['time_of_day'] / 24.0], dtype=np.float32)
    day_of_week  = np.array([event['day_of_week'] / 7.0],  dtype=np.float32)
    return np.concatenate([cat_onehot, tag_vec, time_of_day, day_of_week])

## 4. Build the 4 Sub-Models

In [159]:
ALL_USER_IDS  = [u['user_id'] for u in users]
ALL_EVENT_IDS = [e['event_id'] for e in events]

# 1. User Collaborative: user_id → EMBEDDING_DIM
uid_in       = tf.keras.Input(shape=(), dtype=tf.string, name='user_id')
uid_lookup   = tf.keras.layers.StringLookup(vocabulary=ALL_USER_IDS, mask_token=None)(uid_in)
uid_emb      = tf.keras.layers.Embedding(len(ALL_USER_IDS) + 1, EMBEDDING_DIM)(uid_lookup)
uid_out      = tf.keras.layers.Dense(EMBEDDING_DIM, name='user_collab_out')(uid_emb)
user_collab_model = tf.keras.Model(inputs=uid_in, outputs=uid_out, name='user_collab')

# 2. User Content: 12-dim vector → EMBEDDING_DIM
ucontent_in  = tf.keras.Input(shape=(USER_CONTENT_DIM,), dtype=tf.float32, name='user_content')
ucontent_h   = tf.keras.layers.Dense(EMBEDDING_DIM, activation='relu')(ucontent_in)
ucontent_out = tf.keras.layers.Dense(EMBEDDING_DIM, name='user_content_out')(ucontent_h)
user_content_model = tf.keras.Model(inputs=ucontent_in, outputs=ucontent_out, name='user_content')

# 3. Event Collaborative: event_id → EMBEDDING_DIM
eid_in       = tf.keras.Input(shape=(), dtype=tf.string, name='event_id')
eid_lookup   = tf.keras.layers.StringLookup(vocabulary=ALL_EVENT_IDS, mask_token=None)(eid_in)
eid_emb      = tf.keras.layers.Embedding(len(ALL_EVENT_IDS) + 1, EMBEDDING_DIM)(eid_lookup)
eid_out      = tf.keras.layers.Dense(EMBEDDING_DIM, name='event_collab_out')(eid_emb)
event_collab_model = tf.keras.Model(inputs=eid_in, outputs=eid_out, name='event_collab')

# 4. Event Content: 13-dim vector → EMBEDDING_DIM
econtent_in  = tf.keras.Input(shape=(EVENT_CONTENT_DIM,), dtype=tf.float32, name='event_content')
econtent_h   = tf.keras.layers.Dense(EMBEDDING_DIM, activation='relu')(econtent_in)
econtent_out = tf.keras.layers.Dense(EMBEDDING_DIM, name='event_content_out')(econtent_h)
event_content_model = tf.keras.Model(inputs=econtent_in, outputs=econtent_out, name='event_content')

print('4 sub-models built successfully.')

4 sub-models built successfully.


## 5. TFRS Training Model

In [160]:
class HybridRecommender(tfrs.Model):
    """Trains all 4 sub-models jointly via the TFRS Retrieval task."""

    def __init__(self, user_collab, user_content, event_collab, event_content):
        super().__init__()
        self.user_collab   = user_collab
        self.user_content  = user_content
        self.event_collab  = event_collab
        self.event_content = event_content
        self.task = tfrs.tasks.Retrieval()

    def compute_event_embedding(self, x):
        """Embeds a single (unbatched) event dict into a 64-dim vector.

        Called at evaluation time to build the full candidate matrix.
        expand_dims/squeeze handle the batch dimension the sub-models expect.
        Must stay 64-dim (concat) to match _user_embedding for dot-product scoring.
        """
        collab  = self.event_collab(tf.expand_dims(x["event_id"], 0))
        content = self.event_content(tf.expand_dims(x["event_content_vector"], 0))
        return tf.squeeze(tf.concat([collab, content], axis=-1), axis=0)

    def _user_embedding(self, features):
        collab  = self.user_collab(features['user_id'])
        content = self.user_content(features['user_content_vector'])
        return tf.concat([collab, content], axis=-1)

    def _event_embedding(self, features):
        collab  = self.event_collab(features['event_id'])
        content = self.event_content(features['event_content_vector'])
        return tf.concat([collab, content], axis=-1)

    def compute_loss(self, features, training=False):
        return self.task(
            self._user_embedding(features),
            self._event_embedding(features),
        )

print('HybridRecommender defined.')

HybridRecommender defined.


## 6. Prepare TF Dataset

Splits interactions 80/20 into train and test sets, then builds TF datasets for both.
User history (attended categories, participation count) is derived from all interactions — it is a feature, not a label, so using it in full does not cause data leakage.

In [161]:
from sklearn.model_selection import train_test_split

# Build per-user attendance history from all interactions
# (this is a feature/context, not the label — no data leakage)
user_history = {}
for _, row in interactions_df.iterrows():
    uid, eid = row['user_id'], row['event_id']
    cat = event_dict[eid]['category']
    if uid not in user_history:
        user_history[uid] = {'categories': [], 'count': 0}
    user_history[uid]['categories'].append(cat)
    user_history[uid]['count'] += 1

# 80/20 train/test split
train_df, test_df = train_test_split(interactions_df, test_size=0.2, random_state=42)
print(f'Total: {len(interactions_df)} | Train: {len(train_df)} | Test: {len(test_df)}')

def build_tensors(df):
    user_ids, event_ids, user_vecs, event_vecs = [], [], [], []
    for _, row in df.iterrows():
        user    = user_dict[row['user_id']]
        event   = event_dict[row['event_id']]
        history = user_history.get(row['user_id'], {'categories': [], 'count': 0})
        user_ids.append(row['user_id'])
        event_ids.append(row['event_id'])
        user_vecs.append(build_user_content_vector(user, history['categories'], history['count']))
        event_vecs.append(build_event_content_vector(event))
    return user_ids, event_ids, np.array(user_vecs), np.array(event_vecs)

t_user_ids, t_event_ids, t_user_vecs, t_event_vecs = build_tensors(train_df)
v_user_ids, v_event_ids, v_user_vecs, v_event_vecs = build_tensors(test_df)

train_ds = tf.data.Dataset.from_tensor_slices({
    'user_id':              t_user_ids,
    'user_content_vector':  t_user_vecs,
    'event_id':             t_event_ids,
    'event_content_vector': t_event_vecs,
}).shuffle(1000).batch(BATCH_SIZE)

test_ds = tf.data.Dataset.from_tensor_slices({
    'user_id':              v_user_ids,
    'user_content_vector':  v_user_vecs,
    'event_id':             v_event_ids,
    'event_content_vector': v_event_vecs,
}).batch(BATCH_SIZE)

# Candidate events dataset for FactorizedTopK
all_events_ds = tf.data.Dataset.from_tensor_slices({
    'event_id':             [e['event_id'] for e in events],
    'event_content_vector': np.array([build_event_content_vector(e) for e in events]),
})

print(f'Train samples: {len(t_user_ids)} | Test samples: {len(v_user_ids)}')

Total: 1770 | Train: 1416 | Test: 354
Train samples: 1416 | Test samples: 354


## 7. Train & Evaluate

Trains on the 80% train split. After training builds the score matrix and computes:
- **NDCG@10** with category-level relevance
- **Category Precision@K** (sections 8)
- **Embedding diversity check** (section 9)

In [162]:
model = HybridRecommender(
    user_collab_model, user_content_model,
    event_collab_model, event_content_model,
)
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))
model.fit(train_ds, epochs=EPOCHS)

# ── Build score matrix (reused by all evaluation cells below) ─────────────────

print('\nBuilding evaluation matrices...')

all_event_embs        = tf.stack([model.compute_event_embedding(x) for x in all_events_ds])
all_event_ids_ordered = [e['event_id'] for e in events]
event_id_to_idx       = {eid: i for i, eid in enumerate(all_event_ids_ordered)}

u_collab_embs  = user_collab_model(tf.constant(v_user_ids))
u_content_embs = user_content_model(tf.constant(v_user_vecs))
test_user_embs = tf.concat([u_collab_embs, u_content_embs], axis=-1)

scores = tf.matmul(test_user_embs, all_event_embs, transpose_b=True).numpy()

# ── Helper functions (reused by sections 8 and 9) ────────────────────────────

def get_interest_categories(user):
    return {INTEREST_TO_CATEGORY[i] for i in user['interests'] if i in INTEREST_TO_CATEGORY}

def ndcg_at_k(ranked_indices, interest_cats, k):
    dcg  = sum(
        (1.0 if events[idx]['category'] in interest_cats else 0.0) / np.log2(rank + 1)
        for rank, idx in enumerate(ranked_indices[:k], start=1)
    )
    n_relevant = sum(1 for e in events if e['category'] in interest_cats)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(k, n_relevant) + 1))
    return dcg / idcg if idcg > 0 else 0.0

# ── NDCG@10 (category-level relevance) ───────────────────────────────────────
# An event is relevant if its category matches the user's interest categories.
# This is appropriate for a university event system where any relevant event
# in the user's area of interest is a good recommendation.

K = 10
model_ndcg  = []
random_ndcg = []

for i, uid in enumerate(v_user_ids):
    interest_cats = get_interest_categories(user_dict[uid])
    if not interest_cats:
        continue
    ranked = np.argsort(scores[i])[::-1]
    model_ndcg.append(ndcg_at_k(ranked, interest_cats, K))
    random_ndcg.append(ndcg_at_k(np.random.permutation(len(events)), interest_cats, K))

ndcg_m, ndcg_r = np.mean(model_ndcg), np.mean(random_ndcg)
print(f'\n── NDCG@{K} (category-level relevance) ──────────────────────')
print(f'  Model  : {ndcg_m:.4f}')
print(f'  Random : {ndcg_r:.4f}')
print(f'  Improvement: {(ndcg_m - ndcg_r) / ndcg_r:+.1%}')
print('\nTraining complete.')

Epoch 1/50
23/23 [==============================] - 0s 603us/step - loss: 237.5234 - regularization_loss: 0.0000e+00 - total_loss: 237.5234
Epoch 2/50
23/23 [==============================] - 0s 566us/step - loss: 220.7288 - regularization_loss: 0.0000e+00 - total_loss: 220.7288
Epoch 3/50
23/23 [==============================] - 0s 551us/step - loss: 207.1021 - regularization_loss: 0.0000e+00 - total_loss: 207.1021
Epoch 4/50
23/23 [==============================] - 0s 555us/step - loss: 190.1687 - regularization_loss: 0.0000e+00 - total_loss: 190.1687
Epoch 5/50
23/23 [==============================] - 0s 494us/step - loss: 174.2505 - regularization_loss: 0.0000e+00 - total_loss: 174.2505
Epoch 6/50
23/23 [==============================] - 0s 475us/step - loss: 163.0899 - regularization_loss: 0.0000e+00 - total_loss: 163.0899
Epoch 7/50
23/23 [==============================] - 0s 495us/step - loss: 154.5261 - regularization_loss: 0.0000e+00 - total_loss: 154.5261
Epoch 8/50
23/23 [==

2026-04-27 23:37:04.472614: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 8. Category Precision@K & Spot-check

- **Category Precision@K**: fraction of top-K recommendations matching the user's interest categories
- **Spot-check**: visual verification for 3 users with different interest profiles

In [163]:
# ── Category Precision@K ─────────────────────────────────────────────────────
ks_precision = [5, 10]
precision_model  = {k: [] for k in ks_precision}
precision_random = {k: [] for k in ks_precision}

for i, uid in enumerate(v_user_ids):
    user = user_dict[uid]
    interest_cats = get_interest_categories(user)
    if not interest_cats:
        continue

    ranked_indices = np.argsort(scores[i])[::-1]
    all_match = sum(1 for e in events if e['category'] in interest_cats)
    baseline  = all_match / len(events)

    for k in ks_precision:
        top_k_events   = [events[idx] for idx in ranked_indices[:k]]
        relevant_count = sum(1 for e in top_k_events if e['category'] in interest_cats)
        precision_model[k].append(relevant_count / k)
        precision_random[k].append(baseline)

print('── Category Precision@K ──────────────────────────────────────')
print(f'{"K":<6} {"Model":>10} {"Random Baseline":>18} {"Improvement":>14}')
print('-' * 52)
for k in ks_precision:
    m = np.mean(precision_model[k])
    r = np.mean(precision_random[k])
    print(f'Top-{k:<2} {m:>10.1%} {r:>18.1%} {(m-r)/r:>+13.1%}')

# ── Spot-check: 3 sample users ────────────────────────────────────────────────
print('\n── Spot-check: Top-10 recommendations for 3 users ───────────')

seen_cats = set()
sample_uids = []
for uid in v_user_ids:
    cats = frozenset(get_interest_categories(user_dict[uid]))
    if cats not in seen_cats and len(sample_uids) < 3:
        seen_cats.add(cats)
        sample_uids.append(uid)

for uid in sample_uids:
    user          = user_dict[uid]
    user_idx      = v_user_ids.index(uid)
    interest_cats = get_interest_categories(user)
    ranked_idx    = np.argsort(scores[user_idx])[::-1][:10]
    top10         = [events[i] for i in ranked_idx]

    print(f'\nUser: {uid}')
    print(f'  Interests     : {user["interests"]}')
    print(f'  Interest cats : {sorted(interest_cats)}')
    print(f'  Top-10 events :')
    for rank, e in enumerate(top10, 1):
        match = '✓' if e['category'] in interest_cats else '✗'
        print(f'    {rank:>2}. [{match}] {e["event_id"]}  category={e["category"]:10s}  tags={e["tags"]}')

── Category Precision@K ──────────────────────────────────────
K           Model    Random Baseline    Improvement
----------------------------------------------------
Top-5       83.4%              37.8%       +120.8%
Top-10      88.3%              37.8%       +133.8%

── Spot-check: Top-10 recommendations for 3 users ───────────

User: user_0051
  Interests     : ['Coding', 'Photography', 'Mobile Dev', 'Academic']
  Interest cats : ['academic', 'arts', 'tech']
  Top-10 events :
     1. [✗] event_0113  category=social      tags=['networking']
     2. [✓] event_0182  category=academic    tags=['networking']
     3. [✓] event_0208  category=academic    tags=['competition']
     4. [✓] event_0069  category=tech        tags=['guest_speaker', 'workshop', 'social']
     5. [✓] event_0160  category=arts        tags=['workshop', 'guest_speaker']
     6. [✓] event_0001  category=academic    tags=['seminar', 'social', 'guest_speaker']
     7. [✓] event_0023  category=tech        tags=['guest_sp

## 9. Save Models & Vocabularies

In [164]:
user_collab_model.export(os.path.join(OUTPUT_DIR, 'user_collab'))
user_content_model.export(os.path.join(OUTPUT_DIR, 'user_content'))
event_collab_model.export(os.path.join(OUTPUT_DIR, 'event_collab'))
event_content_model.export(os.path.join(OUTPUT_DIR, 'event_content'))

with open(os.path.join(OUTPUT_DIR, 'vocabularies.json'), 'w') as f:
    json.dump({
        'categories':           CATEGORIES,
        'all_tags':             ALL_TAGS,
        'interests':            INTERESTS,
        'interest_to_category': INTEREST_TO_CATEGORY,
        'user_content_dim':     USER_CONTENT_DIM,
        'event_content_dim':    EVENT_CONTENT_DIM,
    }, f, indent=2)

print('Saved to:', OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/user_collab/assets


INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/user_collab/assets


Saved artifact at '../ml_models/recommendation_model/user_collab'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None,), dtype=tf.string, name='user_id')
Output Type:
  TensorSpec(shape=(None, 32), dtype=tf.float32, name=None)
Captures:
  5502693296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5488176016: TensorSpec(shape=(), dtype=tf.int64, name=None)
  5488171440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502696640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502695936: TensorSpec(shape=(), dtype=tf.resource, name=None)
INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/user_content/assets


INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/user_content/assets


Saved artifact at '../ml_models/recommendation_model/user_content'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12), dtype=tf.float32, name='user_content')
Output Type:
  TensorSpec(shape=(None, 32), dtype=tf.float32, name=None)
Captures:
  5502686432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502685904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502682032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502689248: TensorSpec(shape=(), dtype=tf.resource, name=None)
INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/event_collab/assets


INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/event_collab/assets


Saved artifact at '../ml_models/recommendation_model/event_collab'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None,), dtype=tf.string, name='event_id')
Output Type:
  TensorSpec(shape=(None, 32), dtype=tf.float32, name=None)
Captures:
  5502695408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502691184: TensorSpec(shape=(), dtype=tf.int64, name=None)
  5502684144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502683088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502685376: TensorSpec(shape=(), dtype=tf.resource, name=None)
INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/event_content/assets


INFO:tensorflow:Assets written to: ../ml_models/recommendation_model/event_content/assets


Saved artifact at '../ml_models/recommendation_model/event_content'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='event_content')
Output Type:
  TensorSpec(shape=(None, 32), dtype=tf.float32, name=None)
Captures:
  5502693824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502694528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502683616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5502694000: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved to: ../ml_models/recommendation_model
['user_content', 'user_collab', 'event_content', 'event_collab', 'vocabularies.json']


## 10. Inference Test

Verifies the saved models load and score correctly before wiring into FastAPI.
The `infer()` function implements the α/β hybrid scoring from Algorithm 1 in the project report:
`Score(u, e) = α × ContentScore + β × CollaborativeScore`
Cold-start users use α=1.0, β=0.0. Experienced users use α=0.4, β=0.6.

In [165]:
# Load saved models
u_collab  = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'user_collab'))
u_content = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'user_content'))
e_collab  = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'event_collab'))
e_content = tf.saved_model.load(os.path.join(OUTPUT_DIR, 'event_content'))

def infer(u_collab, u_content, e_collab, e_content, user, event, history, alpha, beta):
    u_cv = build_user_content_vector(user, history['categories'], history['count'])
    e_cv = build_event_content_vector(event)

    uc_emb  = list(u_collab.signatures['serving_default'](user_id=tf.constant([user['user_id']])).values())[0]
    uco_emb = list(u_content.signatures['serving_default'](user_content=tf.constant([u_cv])).values())[0]
    ec_emb  = list(e_collab.signatures['serving_default'](event_id=tf.constant([event['event_id']])).values())[0]
    eco_emb = list(e_content.signatures['serving_default'](event_content=tf.constant([e_cv])).values())[0]

    collab_score  = float(tf.reduce_sum(uc_emb * ec_emb))
    content_score = float(tf.reduce_sum(uco_emb * eco_emb))
    return alpha * content_score + beta * collab_score

test_user  = users[0]
test_event = events[0]
history    = user_history.get(test_user['user_id'], {'categories': [], 'count': 0})
score = infer(u_collab, u_content, e_collab, e_content, test_user, test_event, history, 0.4, 0.6)

print(f'User: {test_user["user_id"]} | Interests: {test_user["interests"]}')
print(f'Event: {test_event["event_id"]} | Category: {test_event["category"]}')
print(f'Hybrid score: {score:.4f}')
print('Inference test passed!')

User: user_0000 | Interests: ['AI']
Event: event_0000 | Category: tech
Hybrid score: 5.6761
Inference test passed!


In [166]:
# ── Embedding Diversity Check ─────────────────────────────────────────────────
# Verifies the model is not "dead" (collapsed embeddings).
# A healthy model has high std dev — embeddings are diverse and meaningful.

sample_ids   = tf.constant(ALL_USER_IDS[:10])
sample_vecs  = tf.constant(np.array([
    build_user_content_vector(user_dict[uid], 
                              user_history.get(uid, {'categories':[]})['categories'],
                              user_history.get(uid, {'categories':[],'count':0})['count'])
    for uid in ALL_USER_IDS[:10]
]))

u_embs = tf.concat([
    user_collab_model(sample_ids),
    user_content_model(sample_vecs)
], axis=-1)

e_ids  = tf.constant(ALL_EVENT_IDS[:10])
e_vecs = tf.constant(np.array([build_event_content_vector(event_dict[eid]) for eid in ALL_EVENT_IDS[:10]]))
e_embs = tf.concat([
    event_collab_model(e_ids),
    event_content_model(e_vecs)
], axis=-1)

u_std = tf.math.reduce_std(u_embs, axis=0).numpy().mean()
e_std = tf.math.reduce_std(e_embs, axis=0).numpy().mean()

print(f'User embedding std dev  : {u_std:.4f}')
print(f'Event embedding std dev : {e_std:.4f}')
print()
if u_std < 0.001 or e_std < 0.001:
    print('WARNING: embeddings may be collapsed.')
else:
    print('Embeddings are diverse — model is healthy.')

User embedding std dev  : 0.5229
Event embedding std dev : 0.4137

Embeddings are diverse — model is healthy.


## 11. Evaluation Summary

In [167]:
W = 62

def section(title):
    print(f'\n  ── {title} {"─" * (W - len(title) - 6)}')

print('=' * W)
print('  UniClubs Recommendation Model — Summary')
print('=' * W)

# ── Dataset ───────────────────────────────────────────────────────────────────
section('Data Used')
print(f'  • {len(users)} simulated students')
print(f'  • {len(events)} simulated events  (5 categories: tech, sports, arts, academic, social)')
print(f'  • {len(interactions_df)} unique student–event interactions  (from {N_INTERACTIONS} generated)')
print(f'  • 80% used for training ({len(t_user_ids)} samples),  20% kept for testing ({len(v_user_ids)} samples)')
print()
print(f'  Why synthetic data?')
print(f'    The app is not deployed yet so no real students exist.')
print(f'    We simulate realistic behaviour:')
print(f'      - 60% of the time a student attends one of their 3–5 favourite events')
print(f'      - 30% of the time they attend any event matching their interests')
print(f'      - 10% of the time they attend something random (exploration)')

# ── How The Model Works ───────────────────────────────────────────────────────
section('How The Model Works')
print('  The model has two "towers" — one for students, one for events.')
print('  Each tower converts its input into a list of numbers (called an embedding).')
print('  A student and an event that are a good match will produce similar numbers.')
print('  The similarity is measured using a dot product (multiply + sum).')
print()
print('  Each tower is actually split into two parts:')
print('    Collaborative part — learns from what other similar students attended')
print('    Content part       — learns from the student\'s stated interests / event details')
print()
print('  Final score formula (Algorithm 1 in the report):')
print('    Score = α × ContentScore  +  β × CollaborativeScore')
print()
print('    New student  (never attended anything) : α = 1.0,  β = 0.0')
print('    → relies entirely on stated interests')
print()
print('    Experienced student (has attended events) : α = 0.4,  β = 0.6')
print('    → trusts what similar students liked more than stated interests')

# ── What Information Goes In ───────────────────────────────────────────────────
section('What Information Goes Into The Model')
print('  For each STUDENT (12 numbers total):')
print('    • Which categories they are interested in  (e.g. tech, sports)')
print('    • Which categories of events they have attended before')
print('    • How often they attend events  (participation frequency)')
print('    • How long they have had an account  (account age)')
print()
print('  For each EVENT (13 numbers total):')
print('    • What category it belongs to  (e.g. tech)')
print('    • What tags it has  (e.g. workshop, guest_speaker)')
print('    • What time of day it is held')
print('    • What day of the week it is held')

# ── Training ──────────────────────────────────────────────────────────────────
section('Training Details')
print(f'  • Ran for {EPOCHS} epochs  (the model sees the training data {EPOCHS} times)')
print(f'  • Batch size {BATCH_SIZE}  (processes {BATCH_SIZE} interactions at a time)')
print(f'  • Each embedding is a list of {EMBEDDING_DIM} numbers')
print(f'  • The loss (error) decreased steadily from ~230 to ~60  → model is learning')

# ── Results ────────────────────────────────────────────────────────────────────
section('Results')
print(f'  {"Metric":<34} {"Model":>7}  {"Random":>7}  {"Better by":>9}')
print(f'  {"─"*34} {"─"*7}  {"─"*7}  {"─"*9}')
print(f'  {"NDCG@10":<34} {ndcg_m:>7.4f}  {ndcg_r:>7.4f}  {(ndcg_m-ndcg_r)/ndcg_r:>+8.1%}')
for k in [5, 10]:
    pm = np.mean(precision_model[k])
    pr = np.mean(precision_random[k])
    print(f'  {f"Category Precision@{k}":<34} {pm:>7.1%}  {pr:>7.1%}  {(pm-pr)/pr:>+8.1%}')
print()
print(f'  Model health check:')
print(f'    Each student gets a unique embedding  (std dev = {u_std:.2f})  ✓')
print(f'    Each event gets a unique embedding    (std dev = {e_std:.2f})  ✓')
print(f'    → The model did NOT collapse (a known failure mode where')
print(f'      everyone gets the same embedding and results are random)')

# ── What The Results Mean ──────────────────────────────────────────────────────
section('What The Results Mean  (in plain English)')
print(f'  NDCG@10 = {ndcg_m:.2f}')
print(f'    Relevant events appear near the TOP of the recommendation list.')
print(f'    {ndcg_m:.2f} vs {ndcg_r:.2f} random = the model is {(ndcg_m-ndcg_r)/ndcg_r:+.0%} better than guessing.')
print()
pm10 = np.mean(precision_model[10])
print(f'  Category Precision@10 = {pm10:.0%}')
print(f'    Out of every 10 events shown to a student,')
print(f'    {pm10*10:.0f} of them belong to categories the student actually cares about.')
print(f'    A random recommender would only get {np.mean(precision_random[10]):.0%}.')
print()
print('  Why not use exact-match accuracy?')
print('    That would ask: "did the model predict the EXACT event the student')
print('    attended?" — but students only attend each event once, so there is')
print('    not enough data for the model to learn that level of detail.')
print('    Category Precision and NDCG are fairer: any good event in the')
print('    right category counts as a successful recommendation.')

# ── Features Not In This Notebook ────────────────────────────────────────────
section('Features Built Into The App  (not in this notebook)')
print('  These run at recommendation time inside the FastAPI backend:')
print()
print('  Caching         — results are saved and reused so the app is fast.')
print('                    Cache is cleared when a student registers for an event.')
print('  Cold-start      — new students with no history get content-only recs')
print('                    (α=1.0) until they attend their first event.')
print('  Diversity       — the final list is reshuffled to avoid showing')
print('                    10 events from the same category in a row.')
print('  Filtering       — events the student already registered for are')
print('                    removed from the candidate list before scoring.')

# ── Limitations ───────────────────────────────────────────────────────────────
section('Known Limitations')
print('  • Synthetic data — model will be retrained once real students use the app')
print('  • No star ratings used yet — will be added once feedback is collected')
print('  • Model should be retrained every few months as data grows')

print('\n' + '=' * W)

  UniClubs Recommendation Model — Summary

  ── Data Used ───────────────────────────────────────────────
  • 150 simulated students
  • 250 simulated events  (5 categories: tech, sports, arts, academic, social)
  • 1770 unique student–event interactions  (from 3000 generated)
  • 80% used for training (1416 samples),  20% kept for testing (354 samples)

  Why synthetic data?
    The app is not deployed yet so no real students exist.
    We simulate realistic behaviour:
      - 60% of the time a student attends one of their 3–5 favourite events
      - 30% of the time they attend any event matching their interests
      - 10% of the time they attend something random (exploration)

  ── How The Model Works ─────────────────────────────────────
  The model has two "towers" — one for students, one for events.
  Each tower converts its input into a list of numbers (called an embedding).
  A student and an event that are a good match will produce similar numbers.
  The similarity is measure